# Pipeline DLT Gold - Pokémon

**Modelagem Dimensional - Dados Pokémon**

Análise de Pokémons, itens, localizações e atributos.

## Dimensões
* dim_pokemon - Pokémons principais
* dim_item - Itens do jogo
* dim_location - Localizações/regiões
* dim_ability - Habilidades
* dim_nature - Naturezas
* dim_growth_rate - Taxas de crescimento

## Bridge Tables
* bridge_pokemon_abilities - Relacionamento N:N entre pokemon e abilities
* bridge_pokemon_locations - Relacionamento N:N entre pokemon e localizações

## Dimensão: Pokémon

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_pokemon
(
  id_pokemon INT COMMENT 'Identificador único do Pokémon (número da Pokédex).',
  nome_pokemon STRING COMMENT 'Nome do Pokémon.',
  CONSTRAINT not_null_id_pokemon EXPECT (id_pokemon IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de Pokémons, identificados por nome e número da Pokédex.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT DISTINCT
  cpn.id AS id_pokemon,
  cpn.pokemon_name AS nome_pokemon
FROM silver_${var.environment}.ds_pokemon.cleaned_pokemon_name cpn
WHERE cpn.id IS NOT NULL;

## Dimensão: Itens

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_item
(
  id_item INT COMMENT 'Identificador único do item.',
  nome_item STRING COMMENT 'Nome do item.',
  custo INT COMMENT 'Custo do item em poké dólares.',
  sprite_url STRING COMMENT 'URL da imagem do sprite do item.',
  CONSTRAINT not_null_id_item EXPECT (id_item IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de itens do jogo Pokémon (poké bolas, medicamentos, TMs, etc.).'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT DISTINCT
  ci.id AS id_item,
  ci.name AS nome_item,
  ci.cost AS custo,
  ci.sprite_url
FROM silver_${var.environment}.ds_pokemon.cleaned_items ci
WHERE ci.id IS NOT NULL;

## Dimensão: Localizações

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_location
(
  id_location INT COMMENT 'Identificador único da localização.',
  nome_location STRING COMMENT 'Nome da localização/região.',
  id_regiao INT COMMENT 'Identificador da região.',
  nome_regiao STRING COMMENT 'Nome da região.',
  CONSTRAINT not_null_id_location EXPECT (id_location IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de localizações e regiões do universo Pokémon.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT DISTINCT
  cl.id AS id_location,
  cl.name AS nome_location,
  cl.region_id AS id_regiao,
  cl.region_name AS nome_regiao
FROM silver_${var.environment}.ds_pokemon.cleaned_locations cl
WHERE cl.id IS NOT NULL;

## Dimensão: Habilidades

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_ability
(
  id_ability INT COMMENT 'Identificador único da habilidade.',
  nome_ability STRING COMMENT 'Nome da habilidade.',
  eh_habilidade_oculta BOOLEAN COMMENT 'Indica se é uma habilidade oculta.',
  CONSTRAINT not_null_id_ability EXPECT (id_ability IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de habilidades (abilities) dos Pokémons.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT DISTINCT
  cpa.ability_id AS id_ability,
  cpa.ability_name AS nome_ability,
  cpa.is_hidden AS eh_habilidade_oculta
FROM silver_${var.environment}.ds_pokemon.cleaned_pokemon_abilities cpa
WHERE cpa.ability_id IS NOT NULL;

## Dimensão: Naturezas

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_nature
(
  id_nature INT COMMENT 'Identificador único da natureza.',
  nome_nature STRING COMMENT 'Nome da natureza.',
  stat_aumentado STRING COMMENT 'Atributo aumentado pela natureza.',
  stat_diminuido STRING COMMENT 'Atributo diminuído pela natureza.',
  CONSTRAINT not_null_id_nature EXPECT (id_nature IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de naturezas dos Pokémons que afetam stats.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT DISTINCT
  cpn.id AS id_nature,
  cpn.nature_name AS nome_nature,
  cpn.increased_stat AS stat_aumentado,
  cpn.decreased_stat AS stat_diminuido
FROM silver_${var.environment}.ds_pokemon.cleaned_pokemon_natures cpn
WHERE cpn.id IS NOT NULL;

## Dimensão: Taxas de Crescimento

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_growth_rate
(
  id_growth_rate INT COMMENT 'Identificador único da taxa de crescimento.',
  nome_growth_rate STRING COMMENT 'Nome da taxa de crescimento (Slow, Medium Fast, Fast, etc.).',
  formula STRING COMMENT 'Fórmula matemática da taxa de crescimento.',
  CONSTRAINT not_null_id_growth_rate EXPECT (id_growth_rate IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de taxas de crescimento que determinam quanto EXP é necessário para subir de nível.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT DISTINCT
  cpgr.id AS id_growth_rate,
  cpgr.growth_rate_name AS nome_growth_rate,
  cpgr.formula
FROM silver_${var.environment}.ds_pokemon.cleaned_pokemon_growth_rates cpgr
WHERE cpgr.id IS NOT NULL;

## Bridge: Pokémon-Habilidades

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW bridge_pokemon_abilities
(
  id_pokemon INT COMMENT 'Identificador do Pokémon (FK).',
  id_ability INT COMMENT 'Identificador da habilidade (FK).',
  eh_habilidade_oculta BOOLEAN COMMENT 'Indica se é habilidade oculta.',
  slot INT COMMENT 'Posição do slot da habilidade (1, 2, 3).'
)
COMMENT 'Bridge table para relacionamento N:N entre Pokémons e habilidades.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT
  cpa.pokemon_id AS id_pokemon,
  cpa.ability_id AS id_ability,
  cpa.is_hidden AS eh_habilidade_oculta,
  cpa.slot
FROM silver_${var.environment}.ds_pokemon.cleaned_pokemon_abilities cpa
WHERE cpa.pokemon_id IS NOT NULL
  AND cpa.ability_id IS NOT NULL;

## Bridge: Pokémon-Localizações

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW bridge_pokemon_locations
(
  id_pokemon INT COMMENT 'Identificador do Pokémon (FK).',
  id_location_area INT COMMENT 'Identificador da área de localização.',
  nome_location_area STRING COMMENT 'Nome da área onde o Pokémon pode ser encontrado.',
  nivel_minimo INT COMMENT 'Nível mínimo de encontro.',
  nivel_maximo INT COMMENT 'Nível máximo de encontro.',
  metodo_encontro STRING COMMENT 'Método de encontro (Walk, Surf, Fish, etc.).',
  taxa_encontro INT COMMENT 'Taxa de encontro (probabilidade).'
)
COMMENT 'Bridge table para relacionamento N:N entre Pokémons e áreas onde podem ser encontrados.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT
  cpla.pokemon_id AS id_pokemon,
  cpla.location_area_id AS id_location_area,
  cpla.location_area_name AS nome_location_area,
  cpla.min_level AS nivel_minimo,
  cpla.max_level AS nivel_maximo,
  cpla.method_name AS metodo_encontro,
  cpla.chance AS taxa_encontro
FROM silver_${var.environment}.ds_pokemon.cleaned_pokemon_location_areas cpla
WHERE cpla.pokemon_id IS NOT NULL;